In [ ]:
import pandas as pd
import numpy as np 
import os

print("Current working directory:", os.getcwd())

In [ ]:
files = {
    "Ba133": "../data/LaBr3 spectrum data/Ba133_calibrated_energy_counts.txt",
    "Co60":  "../data/LaBr3 spectrum data/Co60_calibrated_energy_counts.txt",
    "Cs137": "../data/LaBr3 spectrum data/Cs137_calibrated_energy_counts.txt",
    "Eu152": "../data/LaBr3 spectrum data/Eu152_calibrated_energy_counts.txt",
    "Na22":  "../data/LaBr3 spectrum data/Na22_calibrated_energy_counts.txt",
}

results = [] 

for isotope, path in files.items():
    if not os.path.exists(path):
        print(f"MISSING FILE for {isotope}: {path}")
        continue

    df = pd.read_csv(path, sep = "\t")
    e = df["Energy_keV"].to_numpy()
    c = df["Counts"].to_numpy()

    steps = np.diff(e)
    is_uniform = np.allclose(steps, steps[0], atol=1e-6)
    step_val = steps[0] if is_uniform else "NON-UNIFORM"

    # Count statistics
    total_counts = c.sum()
    max_counts = c.max()
    zero_frac = round((c == 0).sum() / len(c), 3)
    low_count_bins = int(((c >0) & (c < 10)).sum())

    results.append({
        "isotope": isotope,
        "min_keV": e.min(),
        "max_keV": e.max(),
        "step_keV": step_val,
        "n_bins": len(e),
        "total_counts": total_counts,
        "max_counts": max_counts,
        "zero_frac": zero_frac,
        "low_count_bins(1-9)": low_count_bins,
    })


summary = pd.DataFrame(results)

print(summary)

In [ ]:
# !pip install spectres

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src.response_matrix import load_response_matrix

matrix = load_response_matrix("../data/response_matrix.csv")

H_d = matrix.h              # shape (6100, 599)
e_meas = matrix.e_meas_keV  # (6100,) — 0.5 to 6099.5 keV
e_true = matrix.e_true_keV  # (599,) — 20 to 6000 keV

print(H_d.shape, e_meas.min(), e_meas.max())

In [ ]:
from src.response_matrix import validate_response_matrix
from dataclasses import asdict

baseline = validate_response_matrix(matrix)

for k, v in asdict(baseline).items():
    print(f"{k}: {v}")

### Testing how spectres works:

In [ ]:
import numpy as np
from spectres import spectres

# Toy H column: 5 old bins, width 1 keV each, centers at 0.5, 1.5, 2.5, 3.5, 4.5
old_wavs = np.array([0.5, 1.5, 2.5, 3.5, 4.5])
old_vals = np.array([0.0, 2.0, 10.0, 3.0, 0.0])    ## mass per bin — density = mass since width = 1

# New grid: 2 bins, width 2 keV each, centers at 1.0 and 3.0
# -> edges [0,2] and [2,4], which land exactly on old bin boundaries (no partial overlap,
#    so this is easy to verify by hand before trusting a messier real case)
new_wavs = np.array([1.0, 3.0])
new_width = 2.0

# Hand calc: new bin [0,2] fully contains old bins at 0.5 and 1.5 -> 0 + 2 = 2
#            new bin [2,4] fully contains old bins at 2.5 and 3.5 -> 10 + 3 = 13

expected = np.array([2.0, 13.0])

density_out = spectres(new_wavs, old_wavs, old_vals, fill=0.0, verbose = False)
mass_out = density_out * new_width 

print("spectres result: ", mass_out)
print("hand-calc expected:", expected)
print("match:", np.allclose(mass_out, expected))

In [ ]:
# One representative file per group

group_A_grid = pd.read_csv(
    "../data/LaBr3 spectrum data/Ba133_calibrated_energy_counts.txt", sep="\t"
)["Energy_keV"].to_numpy()

group_B_grid = pd.read_csv(
    "../data/LaBr3 spectrum data/Co60_calibrated_energy_counts.txt", sep="\t"
)["Energy_keV"].to_numpy()

group_A_step = 1.729062
group_B_step = 1.783802

In [ ]:
from spectres import spectres

def rebin_H(H, e_meas, new_grid, new_step):
    """
    Downsamples H's E_meas axis (rows) onto new_grid, leaving E_true (columns)
    untouched. Returns H reshaped to (len(new_grid), H.shape[1]).
    """
    H_T = H.T   # shape (n_true, n_meas) — SpectRes needs the resampled axis last

    # fill=0.0: physically correct for any new bin with no detector response

    density = spectres(new_grid, e_meas, H_T, fill=0.0, verbose=False)

    # density -> mass: SpectRes returns per-keV density: without this multiply,
    # every value is wrong by a factor of new_step.
    mass = density * new_step

    return mass.T   # back to (n_new, n_true)

H_groupA = rebin_H(H_d, e_meas, group_A_grid, group_A_step)
H_groupB = rebin_H(H_d, e_meas, group_B_grid, group_B_step)

colsum_A = H_groupA.sum(axis=0)
colsum_B = H_groupB.sum(axis=0)

print("Group A column sums — min/max:", colsum_A.min(), colsum_A.max())
print("Group B column sums — min/max:", colsum_B.min(), colsum_B.max())

In [ ]:
worst_A = np.argsort(colsum_A)[:10]
print("Group A worst columns — E_true (keV):", e_true[worst_A])
print("Group A worst columns — sums:", colsum_A[worst_A])

worst_B = np.argsort(colsum_B)[:10]
print("Group B worst columns — E_true (keV):", e_true[worst_B])
print("Group B worst columns — sums:", colsum_B[worst_B])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(e_true, colsum_A)
axes[0].axhline(0.98, color="red", linestyle="--", label="0.98 threshold")
axes[0].set_xlabel("E_true (keV)")
axes[0].set_ylabel("Column Sum")
axes[0].set_title("Group A")
axes[0].legend()

axes[1].plot(e_true, colsum_B)
axes[1].axhline(0.98, color="red", linestyle="--", label="0.98 threshold")
axes[1].set_xlabel("E_true (keV)")
axes[1].set_ylabel("Column Sum")
axes[1].set_title("Group B")
axes[1].legend()

plt.tight_layout()
plt.show()

# Find the E_true where the sum first drops below 0.98, scanning from low energy up
def first_bad_energy(colsum, e_true, threshold=0.98):
    bad = np.where(colsum < threshold)[0]
    return e_true[bad[0]] if len(bad) > 0 else None

print("Group A first drops below 0.98 at E_true =", first_bad_energy(colsum_A, e_true))
print("Group B first drops below 0.98 at E_true =", first_bad_energy(colsum_B, e_true))

In [ ]:
# Standard calibration-source gamma lines (keV) — well-established nuclear
# decay data, not something that changes; worth a quick check against NNDC
# certificate values later for the final table, but fine for this diagnostic.
isotope_lines = {
    "Ba133": [81.0, 276.4, 302.9, 356.0, 383.8],
    "Cs137": [661.7],
    "Na22":  [511.0, 1274.5],
    "Co60":  [1173.2, 1332.5],
    "Eu152": [121.8, 344.3, 778.9, 964.1, 1112.1, 1408.0],
}

group_map = {
    "Ba133": ("A", colsum_A), "Cs137": ("A", colsum_A), "Na22": ("A", colsum_A),
    "Co60":  ("B", colsum_B), "Eu152": ("B", colsum_B),
}

for isotope, lines in isotope_lines.items():
    group_label, colsum = group_map[isotope]
    print(f"\n{isotope} (Group {group_label}):")
    for line_energy in lines:
        idx = np.argmin(np.abs(e_true - line_energy))  # nearest E_true bin
        print(f"  {line_energy} keV -> nearest E_true={e_true[idx]:.0f} keV, column_sum={colsum[idx]:.4f}")

In [ ]:
# Does the "missing" mass concentrate below each grid's floor —
# i.e., in the region flagged in your prior audit as the E_meas≈0 anomaly?

grid_A_floor = group_A_grid.min() - group_A_step / 2
grid_B_floor = group_B_grid.min() - group_B_step / 2
meas_below_A = e_meas < grid_A_floor
meas_below_B = e_meas < grid_B_floor

print(f"Group A floor {grid_A_floor:.2f} keV -> excludes {meas_below_A.sum()} of {len(e_meas)} H_d rows")
print(f"Group B floor {grid_B_floor:.2f} keV -> excludes {meas_below_B.sum()} of {len(e_meas)} H_d rows")

for label, energy, is_below in [
    ("Na22 1274.5 keV (Group A)", 1274.5, meas_below_A),
    ("Co60 1332.5 keV (Group B)", 1332.5, meas_below_B),
]:
    idx = np.argmin(np.abs(e_true - energy))
    col = H_d[:, idx]
    below_frac = col[is_below].sum() / col.sum()
    print(f"{label}: {100*below_frac:.1f}% of original column mass sits below the grid floor")
    

In [ ]:
# Does Group B's missing mass sit ABOVE the grid's ceiling instead?
# (Also worth checking Group A here for completeness — we only confirmed
# its floor, never checked whether its ceiling contributes anything too.)

grid_A_ceiling = group_A_grid.max() + group_A_step / 2
grid_B_ceiling = group_B_grid.max() + group_B_step / 2
meas_above_A = e_meas > grid_A_ceiling
meas_above_B = e_meas > grid_B_ceiling

for label, energy, is_above, group in [
    ("Na22 1274.5 keV (Group A)", 1274.5, meas_above_A, "A"),
    ("Co60 1332.5 keV (Group B)", 1332.5, meas_above_B, "B"),
]:
    idx = np.argmin(np.abs(e_true - energy))
    col = H_d[:, idx]
    above_frac = col[is_above].sum() / col.sum()
    print(f"{label}: {100*above_frac:.1f}% of original column mass sits above the grid ceiling")

In [ ]:
def manual_rebin_column(col, old_centers, old_step, new_centers, new_step):
    """
    Pure overlap-fraction rebin for ONE column — no library, fully traceable.
    Redistributes each old bin's value across every new bin it overlaps,
    weighted by the fraction of the old bin's width that falls inside
    each new bin (same principle as spectres, just written out explicitly).
    """
    old_left, old_right = old_centers - old_step / 2, old_centers + old_step / 2
    new_left, new_right = new_centers - new_step / 2, new_centers + new_step / 2

    new_vals = np.zeros(len(new_centers))
    for k in range(len(new_centers)):
        overlap = np.minimum(old_right, new_right[k]) - np.maximum(old_left, new_left[k])
        overlap = np.clip(overlap, 0, None)          # negative = no overlap at all
        frac = overlap / old_step                     # share of each old bin assigned to bin k
        new_vals[k] = np.sum(frac * col)
    return new_vals

# Test on the exact problem column: Co60's 1332.5 keV line
idx = np.argmin(np.abs(e_true - 1332.5))
col = H_d[:, idx]

manual_vals = manual_rebin_column(col, e_meas, 1.0, group_B_grid, group_B_step)
spectres_vals = spectres(group_B_grid, e_meas, col, fill=0.0, verbose=False) * group_B_step

print("Original column sum (before any rebin):", col.sum())
print("Manual overlap-rebin sum:", manual_vals.sum())
print("spectres rebin sum:", spectres_vals.sum())
print("Max abs difference, manual vs spectres, per-bin:", np.max(np.abs(manual_vals - spectres_vals)))

In [ ]:
def build_overlap_weights(old_centers, old_step, new_centers, new_step):
    """
    Builds the (n_new x n_old) overlap-fraction matrix: W[k, i] = fraction of
    old bin i's width that falls inside new bin k. Same principle as
    manual_rebin_column, generalized across every bin pair at once.
    """
    old_left, old_right = old_centers - old_step / 2, old_centers + old_step / 2
    new_left, new_right = new_centers - new_step / 2, new_centers + new_step / 2

    # Broadcasting: old_* as row vectors (1, n_old), new_* as column vectors (n_new, 1)
    overlap = np.minimum(old_right[None, :], new_right[:, None]) - \
              np.maximum(old_left[None, :], new_left[:, None])
    overlap = np.clip(overlap, 0, None)  # negative = no overlap
    return overlap / old_step             # (n_new, n_old)

W_A = build_overlap_weights(e_meas, 1.0, group_A_grid, group_A_step)
W_B = build_overlap_weights(e_meas, 1.0, group_B_grid, group_B_step)

H_groupA = W_A @ H_d   # (n_new_A, n_old) @ (n_old, n_true) -> (n_new_A, n_true)
H_groupB = W_B @ H_d

# Re-verify against the column we already trust, before checking anything else
idx = np.argmin(np.abs(e_true - 1332.5))
print("Vectorized result for 1332.5 keV column sum:", H_groupB[:, idx].sum())
print("(should match the manual loop version: 0.9999993979799816)")

colsum_A = H_groupA.sum(axis=0)
colsum_B = H_groupB.sum(axis=0)
print("Group A column sums — min/max:", colsum_A.min(), colsum_A.max())
print("Group B column sums — min/max:", colsum_B.min(), colsum_B.max())

In [ ]:
# Standard calibration-source gamma lines (keV) — well-established nuclear
# decay data, not something that changes; worth a quick check against NNDC
# certificate values later for the final table, but fine for this diagnostic.
isotope_lines = {
    "Ba133": [81.0, 276.4, 302.9, 356.0, 383.8],
    "Cs137": [661.7],
    "Na22":  [511.0, 1274.5],
    "Co60":  [1173.2, 1332.5],
    "Eu152": [121.8, 344.3, 778.9, 964.1, 1112.1, 1408.0],
}

group_map = {
    "Ba133": ("A", colsum_A), "Cs137": ("A", colsum_A), "Na22": ("A", colsum_A),
    "Co60":  ("B", colsum_B), "Eu152": ("B", colsum_B),
}

for isotope, lines in isotope_lines.items():
    group_label, colsum = group_map[isotope]
    print(f"\n{isotope} (Group {group_label}):")
    for line_energy in lines:
        idx = np.argmin(np.abs(e_true - line_energy))  # nearest E_true bin
        print(f"  {line_energy} keV -> nearest E_true={e_true[idx]:.0f} keV, column_sum={colsum[idx]:.4f}")